# 02 - Binarização de Máscaras

Aplica binarização cientificamente validada (Gaussian Blur + Morphological Opening) nas máscaras de Ground Truth e dos modelos de segmentação.

## Imports

In [ ]:
import sys
from pathlib import Path

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import os
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter, binary_opening

from src.io.indice_loader import carregar_indice_excel
from src.io.path_utils import caminho_ground_truth
from src.config import (
    BINARIZATION_KERNEL_SIZE,
    BINARIZATION_SIGMA,
    BINARIZATION_THRESHOLD,
    GROUND_TRUTH_BINARY,
    MODELOS_PARA_AVALIACAO,
    PREDICTED_MASKS_DIR,
    PREDICTED_MASKS_BINARY,
)

## Leitura do índice

Carrega o arquivo Excel com o índice das imagens a serem processadas.

In [ ]:
indice_excel = carregar_indice_excel()

## Verificação de integridade dos PNGs raw

Valida todos os arquivos .png em `generated/predicted_masks/` antes de processar. Remove arquivos corrompidos se encontrados.

In [ ]:
# Verifica integridade dos PNGs em generated/predicted_masks/ e remove arquivos corrompidos
def verificar_e_limpar_pngs_corrompidos(diretorio_base):
    total_png = 0
    arquivos_integros = 0
    arquivos_removidos = 0
    falhas_remocao = 0

    for raiz, _, arquivos in os.walk(diretorio_base):
        for nome_arquivo in arquivos:
            if not nome_arquivo.lower().endswith('.png'):
                continue

            total_png += 1
            caminho_arquivo = os.path.join(raiz, nome_arquivo)

            try:
                with Image.open(caminho_arquivo) as img:
                    img.verify()

                with Image.open(caminho_arquivo) as img:
                    img.load()

                arquivos_integros += 1
            except Exception as e:
                print(f"Corrompido detectado: {caminho_arquivo}")
                print(f" - Erro: {e}")

                try:
                    os.remove(caminho_arquivo)
                    arquivos_removidos += 1
                    print(" - Arquivo removido com sucesso.")
                except OSError as erro_remocao:
                    falhas_remocao += 1
                    print(f" - Falha ao remover arquivo: {erro_remocao}")

    print('\nVerificacao de integridade concluida.')
    print(f" - Total de PNGs verificados: {total_png}")
    print(f" - Arquivos integros: {arquivos_integros}")
    print(f" - Arquivos removidos por corrupcao: {arquivos_removidos}")
    print(f" - Falhas ao remover: {falhas_remocao}")

if os.path.isdir(PREDICTED_MASKS_DIR):
    verificar_e_limpar_pngs_corrompidos(PREDICTED_MASKS_DIR)
else:
    print(f"AVISO: Diretório raw não encontrado: {PREDICTED_MASKS_DIR}")
    print("Execute o notebook 01 primeiro para gerar as máscaras raw.")

## Conversão do Ground Truth para PNG Binarizado

Converte máscaras Ground Truth de JPG para PNG binário usando o método Gaussian Blur + Morphological Opening.

### Como Funciona o Método

O método combina três operações de processamento de imagem para produzir máscaras binárias de alta qualidade:

**1. Gaussian Blur (Filtro Gaussiano)**
- Aplica uma convolução gaussiana com sigma=1.0 sobre a imagem em escala de cinza
- Suaviza transições abruptas entre pixels, removendo artefatos de compressão JPG
- Cada pixel é substituído por uma média ponderada de seus vizinhos, usando pesos gaussianos
- Resultado: imagem com bordas mais suaves e menos ruído de alta frequência

**2. Threshold (Limiarização)**
- Compara cada pixel com o limiar de 127 (ponto médio da escala 0-255)
- Pixels com valor > 127 são classificados como foreground (True/branco)
- Pixels com valor ≤ 127 são classificados como background (False/preto)
- Resultado: imagem booleana binária (apenas dois valores possíveis)

**3. Morphological Opening (Abertura Morfológica)**
- Operação morfológica composta: erosão seguida de dilatação
- Usa elemento estruturante 3×3 (matriz quadrada de True)
- **Erosão**: remove pixels isolados e adelgaça bordas (pixel só permanece se toda vizinhança 3×3 for foreground)
- **Dilatação**: restaura tamanho das regiões e suaviza contornos (pixel vira foreground se qualquer vizinho 3×3 for foreground)
- Resultado: remove ruído pequeno mantendo estruturas principais, produz bordas mais suaves

Parâmetros configurados em `src/config.py`.

### Implementação - Ground Truth

Executa o pipeline de binarização e salva resultados em `generated/ground_truth_binary/*.png`.

In [ ]:
# ===== CONVERSÃO =====
output_dir = GROUND_TRUTH_BINARY
os.makedirs(output_dir, exist_ok=True)

print("Convertendo Ground Truth para PNG binarizado")
print(f"Método: Gaussian Blur + Morphological Opening")
print(f"Parâmetros: SIGMA={BINARIZATION_SIGMA}, LIMIAR={BINARIZATION_THRESHOLD}, KERNEL_SIZE={BINARIZATION_KERNEL_SIZE}")
print(f"Total de imagens: {len(indice_excel)}\n")

processadas = 0
puladas = 0
erros = 0

for idx, linha in enumerate(indice_excel, 1):
    caminho_original = caminho_ground_truth(linha.nome_arquivo)
    caminho_saida = os.path.join(output_dir, f"{linha.nome_arquivo}.png")
    
    if not os.path.isfile(caminho_original):
        print(f"[{idx}/{len(indice_excel)}] ERRO: Máscara não encontrada: {caminho_original}")
        erros += 1
        continue
    
    if os.path.isfile(caminho_saida):
        puladas += 1
        continue
    
    try:
        with Image.open(caminho_original) as img:
            img_gray = img.convert('L')
            matriz = np.array(img_gray, dtype=np.float32)
            
            matriz_blur = gaussian_filter(matriz, sigma=BINARIZATION_SIGMA)
            matriz_binaria = (matriz_blur > BINARIZATION_THRESHOLD).astype(bool)
            
            estrutura = np.ones((BINARIZATION_KERNEL_SIZE, BINARIZATION_KERNEL_SIZE), dtype=bool)
            matriz_opening = binary_opening(matriz_binaria, structure=estrutura)
            
            matriz_final = (matriz_opening * 255).astype(np.uint8)
            img_binarizada = Image.fromarray(matriz_final, mode='L')
            img_binarizada.save(caminho_saida, format='PNG')
        
        processadas += 1
        if processadas % 10 == 0 or processadas == len(indice_excel):
            print(f"[{idx}/{len(indice_excel)}] Processadas: {processadas} | Puladas: {puladas}")
    
    except Exception as e:
        print(f"[{idx}/{len(indice_excel)}] ERRO ao processar {linha.nome_arquivo}: {e}")
        erros += 1

print(f"\n{'='*60}")
print("Conversão CONCLUÍDA")
print(f"Processadas: {processadas} | Puladas: {puladas} | Erros: {erros}")
print(f"Salvo em: {output_dir}")
print(f"{'='*60}")

## Binarização das Máscaras dos Modelos

Aplica o mesmo método de binarização nas máscaras geradas pelos modelos de segmentação para garantir consistência na comparação.

### Pipeline de Processamento

O mesmo processo aplicado ao Ground Truth é executado em todas as máscaras dos modelos:

1. **Leitura**: Carrega máscara raw em escala de cinza (0-255) de `generated/predicted_masks/<modelo>/`

2. **Gaussian Filter**: Aplica blur gaussiano (sigma=1.0) para suavizar transições e reduzir ruído de alta frequência

3. **Threshold**: Binariza usando limiar 127 - separa foreground (>127) de background (≤127)

4. **Morphological Opening**: Aplica erosão + dilatação com kernel 3×3 para remover pixels isolados e suavizar bordas

5. **Salvamento**: Converte resultado booleano para uint8 (0 ou 255) e salva em `generated/predicted_masks_binary/<modelo>/`

Usar o mesmo método em todas as máscaras garante que diferenças nas métricas reflitam apenas o desempenho dos modelos, não variações no pré-processamento.

In [ ]:
print("Binarizando máscaras dos modelos")
print(f"Método: Gaussian Blur + Morphological Opening")
print(f"Parâmetros: SIGMA={BINARIZATION_SIGMA}, LIMIAR={BINARIZATION_THRESHOLD}, KERNEL_SIZE={BINARIZATION_KERNEL_SIZE}\n")

for modelo in MODELOS_PARA_AVALIACAO.keys():
    print(f"\n{'='*60}")
    print(f"Modelo: {modelo}")
    print(f"{'='*60}")
    
    input_dir = os.path.join(PREDICTED_MASKS_DIR, modelo)
    output_dir = os.path.join(PREDICTED_MASKS_BINARY, modelo)
    os.makedirs(output_dir, exist_ok=True)
    
    if not os.path.isdir(input_dir):
        print(f"AVISO: Diretório raw não encontrado: {input_dir}")
        print(f"Execute o notebook 01 primeiro para gerar as máscaras raw.")
        continue
    
    processadas = 0
    puladas = 0
    erros = 0
    
    for idx, linha in enumerate(indice_excel, 1):
        caminho_entrada = os.path.join(input_dir, f"{linha.nome_arquivo}.png")
        caminho_saida = os.path.join(output_dir, f"{linha.nome_arquivo}.png")
        
        if not os.path.isfile(caminho_entrada):
            print(f"[{idx}/{len(indice_excel)}] ERRO: Máscara raw não encontrada: {caminho_entrada}")
            erros += 1
            continue
        
        if os.path.isfile(caminho_saida):
            puladas += 1
            continue
        
        try:
            with Image.open(caminho_entrada) as img:
                if img.mode != 'L':
                    img = img.convert('L')
                
                matriz = np.array(img, dtype=np.float32)
                
                matriz_blur = gaussian_filter(matriz, sigma=BINARIZATION_SIGMA)
                matriz_binaria = (matriz_blur > BINARIZATION_THRESHOLD).astype(bool)
                
                estrutura = np.ones((BINARIZATION_KERNEL_SIZE, BINARIZATION_KERNEL_SIZE), dtype=bool)
                matriz_opening = binary_opening(matriz_binaria, structure=estrutura)
                
                matriz_final = (matriz_opening * 255).astype(np.uint8)
                img_binarizada = Image.fromarray(matriz_final, mode='L')
                img_binarizada.save(caminho_saida, format='PNG')
            
            processadas += 1
            if processadas % 10 == 0 or processadas == len(indice_excel):
                print(f"[{modelo}] Processadas: {processadas} | Puladas: {puladas}")
        
        except Exception as e:
            print(f"[{idx}/{len(indice_excel)}] ERRO ao processar {linha.nome_arquivo}: {e}")
            erros += 1
    
    print(f"\n{modelo} - Processadas: {processadas} | Puladas: {puladas} | Erros: {erros}")

print(f"\n{'='*60}")
print("Binarização de todos os modelos CONCLUÍDA")
print(f"Máscaras salvas em: {PREDICTED_MASKS_BINARY}")
print(f"{'='*60}")